In [1]:
from utils.evaluation.scoring_func import get_chem
import numpy as np
from rdkit import Chem
import torch
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*') 

/home/east/.conda/envs/pharm/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_resgen_results(ckpt_path, reference_list):
    input_dicts = torch.load(ckpt_path)['all_results']

    for pr in input_dicts:
            pr['ligand_filename'] = pr['ligand_filename'].replace("_gen", "")

    all_ligand_filenames = set([pr['ligand_filename'] for pr in input_dicts])

    all_ligand_filenames = sorted(list(all_ligand_filenames) + [item for item in reference_list if item not in all_ligand_filenames], 
                     key=lambda x: reference_list.index(x))

    categorized_dicts = {category: [] for category in all_ligand_filenames}

    for d in input_dicts:
        unique_key = d['ligand_filename']
        categorized_dicts[unique_key].append(d)
    
    for key in reference_list:
        if key not in categorized_dicts:
            categorized_dicts[key] = []

    for pr in categorized_dicts:
        categorized_dicts[pr] = categorized_dicts[pr][:100]

    categorized_dicts = [categorized_dicts[pr] for pr in categorized_dicts]

    return categorized_dicts

def load_results(ckpt_path, reference_list):
    input_dicts = torch.load(ckpt_path)
    all_ligand_filenames = set([pr['ligand_filename'] for pr in input_dicts])
    all_ligand_filenames = sorted(all_ligand_filenames, key=lambda x: reference_list.index(x))
    all_ligand_filenames = sorted(list(all_ligand_filenames) + [item for item in reference_list if item not in all_ligand_filenames], 
                     key=lambda x: reference_list.index(x))

    categorized_dicts = {category: [] for category in all_ligand_filenames}
    for d in input_dicts:
        unique_key = d['ligand_filename']
        categorized_dicts[unique_key].append(d)

    for key in reference_list:
        if key not in categorized_dicts:
            categorized_dicts[key] = []

    categorized_dicts = [categorized_dicts[pr] for pr in categorized_dicts]
    return categorized_dicts

# Load Data

In [3]:
class Globals:
    reference_path = 'eval_results/crossdocked_test_vina_docked_pose_checked.pt'
    split_pocket_path = 'eval_results/cvae_vina_docked.pt'

    ar_path = 'eval_results/ar_vina_docked_pose_checked.pt'
    pocket2mol_path = 'eval_results/pocket2mol_vina_docked_pose_checked.pt'
    targetdiff_path = 'eval_results/targetdiff_vina_docked_pose_checked.pt'
    flag_path = 'eval_results/flag_vina_docked_pose_checked.pt'
    resgen_path = 'eval_results/resgen_vina_docked_pose_checked.pt'
    decompdiff_path = 'eval_results/decompdiff_vina_docked_pose_checked.pt'
    molcraft_path = 'eval_results/molcraft_vina_docked_pose_checked.pt'
    ipdiff_path = 'eval_results/ipdiff_vina_docked_pose_checked.pt'
    ours_path = 'eval_results/sefmol_vina_docked_pose_checked.pt'

In [4]:
Globals.reference_results = torch.load(Globals.reference_path)
Globals.reference_results = [[v] for v in Globals.reference_results]
split_pocket = torch.load(Globals.split_pocket_path)
split_pocket_list = [i[0]['ligand_filename'] for i in split_pocket]

Globals.ar_results = load_results(Globals.ar_path, split_pocket_list)
Globals.pocket2mol_results = load_results(Globals.pocket2mol_path, split_pocket_list)
Globals.flag_results = load_results(Globals.flag_path, split_pocket_list)
Globals.resgen_results = load_resgen_results(Globals.resgen_path, split_pocket_list)
Globals.targetdiff_results = load_results(Globals.targetdiff_path, split_pocket_list)
Globals.decompdiff_results = load_results(Globals.decompdiff_path, split_pocket_list)
Globals.molcraft_results = load_results(Globals.molcraft_path, split_pocket_list)
Globals.ipdiff_results = load_results(Globals.ipdiff_path, split_pocket_list)
Globals.ours_results = load_results(Globals.ours_path, split_pocket_list)

# Metrics Summary

In [5]:
def print_results(results, show_vina=True):
    qed = [r['chem_results']['qed'] for r in results]
    sa = [r['chem_results']['sa'] for r in results]
    
    try:
        logp = [r['chem_results']['logp'] for r in results]
        tpsa = [r['chem_results']['tpsa'] for r in results]
        fsp3 = [r['chem_results']['fsp3'] for r in results]
        hba = [r['chem_results']['hba'] for r in results]
        hbd = [r['chem_results']['hbd'] for r in results]
        rotb = [r['chem_results']['rotb'] for r in results]
        lipinski = [r['chem_results']['lipinski'] for r in results]
    except:
        chem_results = [get_chem(r['mol']) for r in results]
        logp = [cr['logp'] for cr in chem_results]
        tpsa = [cr['tpsa'] for cr in chem_results]
        fsp3 = [cr['fsp3'] for cr in chem_results]
        hba = [cr['hba'] for cr in chem_results]
        hbd = [cr['hbd'] for cr in chem_results]
        rotb = [cr['rotb'] for cr in chem_results]
        lipinski = [cr['lipinski'] for cr in chem_results]
     

    mol_size = [r['mol'].GetNumAtoms() for r in results]
    print('Num results: %d' % len(results))

    if show_vina:
        vina_score_only = [x['vina']['score_only'][0]['affinity'] for x in results]
        vina_min = [x['vina']['minimize'][0]['affinity'] for x in results]
        vina_dock = [r['vina']['dock'][0]['affinity'] for r in results]
        
        count = 0

        for i, r in enumerate(results):
             if logp[i] >= -0.4 and \
                logp[i] <= 5.6 and \
                tpsa[i] <= 140 and \
                fsp3[i] >= 0.42 and \
                hba[i] <= 10 and \
                hbd[i] <= 5 and \
                rotb[i] <= 10 and \
                r['vina']['dock'][0]['affinity'] < -8.18 and \
                r['chem_results']['qed'] > 0.25 and \
                r['chem_results']['sa'] > 0.59:
                    count += 1
        sr_score = count / len(results)
    
        print('[Vina Score] Avg: %.2f | Med: %.2f' % (np.mean(vina_score_only), np.median(vina_score_only)))
        print('[Vina Min]   Avg: %.2f | Med: %.2f' % (np.mean(vina_min), np.median(vina_min)))
        print('[Vina Dock]  Avg: %.2f | Med: %.2f' % (np.mean(vina_dock), np.median(vina_dock)))
        
    print('[QED]  Avg: %.2f | Med: %.2f' % (np.mean(qed), np.median(qed)))
    print('[SA]   Avg: %.2f | Med: %.2f' % (np.mean(sa), np.median(sa)))
    print('[lipinski]   Avg: %.2f | Med: %.2f' % (np.mean(lipinski), np.median(lipinski)))
    print('[logp]   Avg: %.2f | Med: %.2f' % (np.mean(logp), np.median(logp)))
    
    print('[SR%%] %.2f%% ' % (sr_score*100, ))
    print('[Size] Avg: %.4f | Med: %.4f' % (np.mean(mol_size), np.median(mol_size)))



def compute_high_affinity(vina_ref, results):
    percentage_good = []
    num_docked = []
    for i in range(100):
        score_ref = vina_ref[i]
        pocket_results = [r for r in results[i] if r['vina'] is not None]

        if len(pocket_results) < 50:
            continue

        num_docked.append(len(pocket_results))

        scores_gen = []
        for docked in pocket_results:
            aff = docked['vina']['dock'][0]['affinity']
            scores_gen.append(aff)

        scores_gen = np.array(scores_gen)
        percentage_good.append((scores_gen <= score_ref).mean())

    percentage_good = np.array(percentage_good)
    num_docked = np.array(num_docked)

    print('[HF%%]  Avg: %.1f%% | Med: %.1f%% ' % (np.mean(percentage_good)*100, np.median(percentage_good)*100))

## Reference

In [6]:
flat_ref_docked = [r for pr in Globals.reference_results for r in pr]
print_results(flat_ref_docked)

Num results: 100
[Vina Score] Avg: -6.36 | Med: -6.46
[Vina Min]   Avg: -6.71 | Med: -6.49
[Vina Dock]  Avg: -7.45 | Med: -7.26
[QED]  Avg: 0.48 | Med: 0.47
[SA]   Avg: 0.73 | Med: 0.74
[lipinski]   Avg: 4.27 | Med: 5.00
[logp]   Avg: 0.89 | Med: 1.09
[SR%] 4.00% 
[Size] Avg: 22.7500 | Med: 21.5000


In [7]:
vina_ref = [r['vina']['dock'][0]['affinity'] for r in flat_ref_docked]

## AR

In [8]:
flat_ar_docked = [r for pr in Globals.ar_results for r in pr]
print_results(flat_ar_docked)
compute_high_affinity(vina_ref, Globals.ar_results)

Num results: 9295
[Vina Score] Avg: -5.75 | Med: -5.64
[Vina Min]   Avg: -6.18 | Med: -5.88
[Vina Dock]  Avg: -6.75 | Med: -6.62
[QED]  Avg: 0.51 | Med: 0.50
[SA]   Avg: 0.64 | Med: 0.63
[lipinski]   Avg: 4.75 | Med: 5.00
[logp]   Avg: 0.39 | Med: 0.33
[SR%] 3.28% 
[Size] Avg: 17.6815 | Med: 16.0000
[HF%]  Avg: 37.9% | Med: 31.0% 


## Pocket2Mol

In [9]:
flat_pocket2mol_docked = [r for pr in Globals.pocket2mol_results for r in pr]
print_results(flat_pocket2mol_docked)
compute_high_affinity(vina_ref, Globals.pocket2mol_results)

Num results: 9831
[Vina Score] Avg: -5.14 | Med: -4.70
[Vina Min]   Avg: -6.42 | Med: -5.82
[Vina Dock]  Avg: -7.15 | Med: -6.79
[QED]  Avg: 0.57 | Med: 0.58
[SA]   Avg: 0.76 | Med: 0.76
[lipinski]   Avg: 4.88 | Med: 5.00
[logp]   Avg: 1.69 | Med: 1.45
[SR%] 2.25% 
[Size] Avg: 17.7363 | Med: 15.0000
[HF%]  Avg: 48.4% | Med: 51.0% 


## FLAG

In [10]:
flat_flag_docked = [r for pr in Globals.flag_results for r in pr]
print_results(flat_flag_docked)
compute_high_affinity(vina_ref, Globals.flag_results)

Num results: 9863
[Vina Score] Avg: 45.98 | Med: 36.62
[Vina Min]   Avg: 6.17 | Med: -2.91
[Vina Dock]  Avg: -5.24 | Med: -5.71
[QED]  Avg: 0.61 | Med: 0.62
[SA]   Avg: 0.63 | Med: 0.62
[lipinski]   Avg: 4.98 | Med: 5.00
[logp]   Avg: 1.61 | Med: 1.38
[SR%] 0.44% 
[Size] Avg: 16.6522 | Med: 16.0000
[HF%]  Avg: 22.7% | Med: 7.0% 


## ResGen

In [11]:
flat_resgen_docked = [r for pr in Globals.resgen_results for r in pr]
print_results(flat_resgen_docked)
compute_high_affinity(vina_ref, Globals.resgen_results)

Num results: 9895
[Vina Score] Avg: 10.50 | Med: 2.54
[Vina Min]   Avg: -2.94 | Med: -4.41
[Vina Dock]  Avg: -6.59 | Med: -6.45
[QED]  Avg: 0.58 | Med: 0.59
[SA]   Avg: 0.78 | Med: 0.79
[lipinski]   Avg: 4.90 | Med: 5.00
[logp]   Avg: 1.68 | Med: 1.52
[SR%] 0.69% 
[Size] Avg: 17.2436 | Med: 15.0000
[HF%]  Avg: 38.0% | Med: 25.0% 


## TargetDiff

In [12]:
flat_targetdiff_docked = [r for pr in Globals.targetdiff_results for r in pr]
print_results(flat_targetdiff_docked)
compute_high_affinity(vina_ref, Globals.targetdiff_results)

Num results: 9036
[Vina Score] Avg: -5.47 | Med: -6.30
[Vina Min]   Avg: -6.64 | Med: -6.83
[Vina Dock]  Avg: -7.80 | Med: -7.91
[QED]  Avg: 0.48 | Med: 0.48
[SA]   Avg: 0.58 | Med: 0.58
[lipinski]   Avg: 4.51 | Med: 5.00
[logp]   Avg: 1.36 | Med: 1.35
[SR%] 4.33% 
[Size] Avg: 24.2371 | Med: 24.0000
[HF%]  Avg: 58.1% | Med: 59.1% 


## Decompdiff

In [13]:
flat_decompdiff_docked = [r for pr in Globals.decompdiff_results for r in pr]
print_results(flat_decompdiff_docked)
compute_high_affinity(vina_ref, Globals.decompdiff_results)

Num results: 7196
[Vina Score] Avg: -5.67 | Med: -6.04
[Vina Min]   Avg: -7.04 | Med: -7.09
[Vina Dock]  Avg: -8.39 | Med: -8.43
[QED]  Avg: 0.45 | Med: 0.43
[SA]   Avg: 0.61 | Med: 0.60
[lipinski]   Avg: 4.31 | Med: 5.00
[logp]   Avg: 2.40 | Med: 2.53
[SR%] 9.82% 
[Size] Avg: 29.4402 | Med: 30.0000
[HF%]  Avg: 64.4% | Med: 71.0% 


## MolCRAFT

In [14]:
flat_molcraft_docked = [r for pr in Globals.molcraft_results for r in pr]
print_results(flat_molcraft_docked)
compute_high_affinity(vina_ref, Globals.molcraft_results)

Num results: 9667
[Vina Score] Avg: -6.59 | Med: -7.04
[Vina Min]   Avg: -7.27 | Med: -7.26
[Vina Dock]  Avg: -7.92 | Med: -8.01
[QED]  Avg: 0.50 | Med: 0.51
[SA]   Avg: 0.69 | Med: 0.68
[lipinski]   Avg: 4.46 | Med: 5.00
[logp]   Avg: 1.16 | Med: 1.28
[SR%] 5.13% 
[Size] Avg: 22.7100 | Med: 23.0000
[HF%]  Avg: 59.1% | Med: 62.6% 


## IPDiff

In [15]:
flat_ipdiff_docked = [r for pr in Globals.ipdiff_results for r in pr]
print_results(flat_ipdiff_docked)
compute_high_affinity(vina_ref, Globals.ipdiff_results)

Num results: 9049
[Vina Score] Avg: -6.66 | Med: -7.47
[Vina Min]   Avg: -7.64 | Med: -7.69
[Vina Dock]  Avg: -8.49 | Med: -8.39
[QED]  Avg: 0.50 | Med: 0.51
[SA]   Avg: 0.56 | Med: 0.56
[lipinski]   Avg: 4.40 | Med: 5.00
[logp]   Avg: 4.00 | Med: 4.06
[SR%] 3.44% 
[Size] Avg: 24.4000 | Med: 25.0000
[HF%]  Avg: 68.5% | Med: 72.2% 


## Ours

In [16]:
flat_ours_docked = [r for pr in Globals.ours_results for r in pr]
print_results(flat_ours_docked)
compute_high_affinity(vina_ref, Globals.ours_results)

Num results: 9829
[Vina Score] Avg: -7.23 | Med: -7.70
[Vina Min]   Avg: -8.03 | Med: -8.00
[Vina Dock]  Avg: -8.72 | Med: -8.75
[QED]  Avg: 0.63 | Med: 0.64
[SA]   Avg: 0.60 | Med: 0.60
[lipinski]   Avg: 4.90 | Med: 5.00
[logp]   Avg: 1.95 | Med: 1.81
[SR%] 11.53% 
[Size] Avg: 22.8823 | Med: 23.0000
[HF%]  Avg: 68.7% | Med: 76.3% 
